# Fitting the Crab Spectrum with the Neural-Network Response and Background Approximation

## Introduction

This notebook provides an overview of the different new classes introduced to allow for a truly unbinned spectral analysis of continuum point sources. The user will mainly interact with:

1. `NFResponse` and `UnpolarizedNFFarFieldInstrumentResponseFunction`: Provide the **C-A-RQS + Spherical Harmonics Expansion** approximation of the response.
2. `NFBackground` and `FreeNormNFUnbinnedBackground`: Provide the **C-A-RQS + Analytical Rate Model** approximation of the simulated background (currently total DC4).
3. `UnbinnedThreeMLPointSourceResponseIRFAdaptive`: Perform the folding of the response with the flux model in a more efficient way.
4. `CachedUnbinnedThreeMLModelFolding`: Adds the capability to save and load the cache of `UnbinnedThreeMLPointSourceResponseIRFAdaptive`.

### Inner Workings

For a comprehensive description of the underlying architecture or more detailed comparison plots and benchmarks, please for now refer to my (Pascal J.) thesis, "Development of an Efficient Response Description for the COSI MeV 𝜸-Ray Telescope." Here, I will just provide a very basic explanation of the code structure.

#### Flow-Based Models

The building blocks managing the approximations are `NFResponse` and `NFBackground` (both of type `NFBase`). During setup, both require a checkpoint file (e.g., `unpolarized_nfresponse_v1-00.pt`) which contains all the necessary information, such as the neural network weights and hyperparameters, the version number to choose the correct model, or the coefficients for the effective area or background rate model.

Another important task is the creation of compute pools. All PyTorch-related inference tasks are managed by a separate process for each chosen device. These processes need to be started or stopped, as shown in the example below.

All queries, such as evaluating the effective area or density as well as sampling, are passed through an `Approximation` object (which chooses the correct model and prepares the input) to a `Model` object (which handles the actual inference on the PyTorch device). For the response $R = A_\mathrm{eff}(\nu\lambda, E_i) \cdot P(E_m, \phi, \psi\chi|\nu\lambda, E_i)$, this includes:

1. The effective area $A_\mathrm{eff}$, modeled with a spherical harmonics expansion. The latter are evaluated using the PyTorch backend of the `sphericart` library.
2. The probability density $P(E_m, \phi, \psi\chi|\nu\lambda, E_i)$, modeled with conditional-autoregressive-rational quadratic spline flows. The library used is called `normflows`.

The background model $B = R(t) \cdot P(E_m, \phi, \psi\chi|t)$ follows a very similar structure and therefore uses most of the same code. While the rate $R(t)$ is always computed on the CPU, the density $P$ is also modeled using `normflows`.

#### Folding

For the unbinned analysis, we need to calculate the total number of events we expect, $N$, and the expectation density, $\mathrm{d}N/\mathrm{d}t\mathrm{d}E_m\mathrm{d}\phi\mathrm{d}\psi\chi$. The latter is especially difficult to compute, since it requires folding the response with the flux model and therefore computing a numerical integral with enough precision for every event in the analysis. The background model $B$, on the other hand, is already the expectation density and requires no integration.

The folding can be performed using `UnbinnedThreeMLPointSourceResponseTrapz`. However, since $R$ has a low inference rate, `UnbinnedThreeMLPointSourceResponseIRFAdaptive` provides an optimized implementation. It uses the fact that many flux models are "well-behaved" (low order in a Taylor series). This means one can significantly reduce the integration error by concentrating most Gauss-Legendre integration nodes at peaks of the response and distributing the remaining ones in between. 

The exact parameters can be tuned by the user, but it probably won't be able to account for very complex spectral shapes. For each integration node, the response is evaluated once and cached, which takes most of the time. This way, during optimization, only the flux model changes and the fitting is fast.

Using `CachedUnbinnedThreeMLModelFolding` instead of `UnbinnedThreeMLModelFolding` adds the capability to save this cache and other parameters of `UnbinnedThreeMLPointSourceResponseIRFAdaptive`. These files can be shared with other scientists and loaded during another session to skip the expensive cache initialization.

### Hardware Requirements

It is highly recommended to use a GPU, preferably by NVIDIA, for the inference. A CPU, even something like an AMD Threadripper, will only allow you to analyze a few orbits in a reasonable time. Also, note that the unbinned analysis requires you to have enough RAM, which increases as the number of events you include in the analysis increases. The batch size can be decreased to limit the maximum consumption.

## Example

### Basic Setup

In [ ]:
from pathlib import Path
from cosipy.util import fetch_wasabi_file
from cosipy.spacecraftfile import SpacecraftHistory
from astropy.time import Time

import astropy.units as u
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np

from threeML import Band, PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter

from cosipy.threeml.unbinned_model_folding import CachedUnbinnedThreeMLModelFolding
from cosipy.statistics import UnbinnedLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.interfaces.expectation_interface import SumExpectationDensity

from cosipy.event_selection.time_selection import TimeSelector
from cosipy.data_io.EmCDSUnbinnedData import TimeTagEmCDSEventDataInSCFrameFromDC3Fits

from cosipy.response.NFResponse import NFResponse
from cosipy.response.nf_instrument_response_function import UnpolarizedNFFarFieldInstrumentResponseFunction

from cosipy.background_estimation.NFBackground import NFBackground
from cosipy.background_estimation.nf_unbinned_background import FreeNormNFUnbinnedBackground

from cosipy.threeml.optimized_unbinned_folding import UnbinnedThreeMLPointSourceResponseIRFAdaptive

In [ ]:
data_path = Path("./data_files/")

crab_data_path = data_path / "crab_standard_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
fetch_wasabi_file('COSI-SMEX/DC3/Data/Sources/crab_standard_3months_unbinned_data_filtered_with_SAAcut.fits.gz', 
                  output=str(crab_data_path))

sc_orientation_path = data_path / "DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
fetch_wasabi_file('COSI-SMEX/DC3/Data/Orientation/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori', 
                  output=str(sc_orientation_path))

bkg_data_path = data_path / "Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz"
fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz', 
                  output=str(bkg_data_path))

rsp_path = data_path / "unpolarized_nfresponse_v1-00.pt"
fetch_wasabi_file('cosi-pipeline-public/COSI-SMEX/DC4/Data/Responses/unpolarized_nfresponse_v1-00.pt',
                  checksum = 'a4f9a7842f2a7345f604da32a155803f', output=str(rsp_path))

bkg_path = data_path / "nfbackground_v1-01.pt"
fetch_wasabi_file('cosi-pipeline-public/COSI-SMEX/DC4/Data/Responses/nfbackground_v1-01.pt',
                  checksum = '52a4fb024930e18430bff80db882d3d5', output=str(bkg_path))

For this tutorial we only use one day of data

In [ ]:
tstart = Time("2028-03-15 00:00:00.000")
tstop = Time("2028-03-15 02:00:00.000")
sc_orientation = SpacecraftHistory.open(sc_orientation_path)
sc_orientation = sc_orientation.select_interval(tstart, tstop)

The time selection is very slow (see issue https://github.com/cositools/cosipy/issues/504). Consider using your own implementation for now.

In [ ]:
data_file = [crab_data_path, bkg_data_path]
selector = TimeSelector(tstart = sc_orientation.tstart, tstop = sc_orientation.tstop)
data = TimeTagEmCDSEventDataInSCFrameFromDC3Fits(data_file, selection=selector)

In [ ]:
print(f"This analysis uses {data.nevents} Events")

`NFResponse` and `NFBackground`
- devices: Optional default devices. For CPU, choose `["cpu"]`.
    - Default devices cause the pool to be initialized and closed automatically for each inference. This can produce unnecessary overhead when not used with functions like `UnbinnedThreeMLPointSourceResponseIRFAdaptive`, which handles this management internally.
    - You can also manually manage the pools using `init_compute_pool`, `clean_compute_pool`, and `shutdown_compute_pool`.
- compile_mode: Optional from a list of options. If issues arise, choose `None`.

In [ ]:
rsp = NFResponse(
    path_to_model=rsp_path,
    area_batch_size=300_000,
    density_batch_size=100_000, 
    devices=["cuda:0", "cuda:1", "cuda:2", "cuda:3"],
    area_compile_mode="max-autotune-no-cudagraphs",
    density_compile_mode="default",
    show_progress=True)

irf = UnpolarizedNFFarFieldInstrumentResponseFunction(rsp)

In [ ]:
bkg_model = NFBackground(
    path_to_model=bkg_path,
    density_batch_size=100_000,
    devices=["cuda:0", "cuda:1", "cuda:2", "cuda:3"],
    density_compile_mode="default",
    show_progress=True)

bkg = FreeNormNFUnbinnedBackground(
    model=bkg_model, 
    data=data, 
    sc_history=sc_orientation, 
    label="bkg_norm")

The next step takes some time, as it needs to interpolate the `SpacecraftHistory` (again issue https://github.com/cositools/cosipy/issues/504)

In [ ]:
psr = UnbinnedThreeMLPointSourceResponseIRFAdaptive(
    data=data, 
    irf=irf, 
    sc_history=sc_orientation, 
    show_progress=True, 
    force_energy_node_caching=True, # Saves the energy nodes even when the batch size is too small
    reduce_memory=True) # May reduce the peak memory consumption (saves the cache as float32). Look out for warnings

# There are several other options that can be set, for example:

psr.integration_batch_size = 100_000

The default `Band`, `Powerlaw` and `PointSource` are very slow (see discussion https://github.com/cositools/cosipy/discussions/492). Consider implementing your own version with torch.

In [ ]:
l = 184.56
b = -5.78

alpha = -1.99
beta = -2.32
xp = 531. * u.keV * (alpha + 2)
piv = 500. * u.keV
K = 3.07e-5 / u.cm / u.cm / u.s / u.keV

spectrum = Band()

spectrum.alpha.min_value = -2.14
spectrum.alpha.max_value = 3.0
spectrum.beta.min_value = -5.0
spectrum.beta.max_value = -2.15
spectrum.xp.min_value = 1.0
spectrum.alpha.delta = 0.01
spectrum.beta.delta = 0.01

spectrum.alpha.value = alpha
spectrum.beta.value = beta
spectrum.xp.value = xp.value
spectrum.K.value = K.value
spectrum.piv.value = piv.value

spectrum.xp.unit = xp.unit
spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit

In [ ]:
spectrum_inj = deepcopy(spectrum)

In [ ]:
source = PointSource("Crab",
                     l=l,
                     b=b,
                     spectral_shape=spectrum)
model = Model(source)

In [ ]:
response = CachedUnbinnedThreeMLModelFolding(psr)

In [ ]:
expectation_density = SumExpectationDensity(response, bkg)

In [ ]:
like_fun = UnbinnedLikelihood(expectation_density)
cosi = ThreeMLPluginInterface('cosi', like_fun, response, bkg)

In [ ]:
bkg_norm = bkg.norm.to_value(u.Hz)

cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                          bkg_norm,
                                          unit = u.Hz,
                                          min_value=0,
                                          max_value=100,
                                          delta=0.05,
                                          )
print(f"The average background flux is {bkg_norm:.2f} Hz")

In [ ]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=True) # You can disable debugging

### Initializing the Cache

Here you could load the cache

In [ ]:
# response.load_caches(data_path / "Crab_tutorial")

The cache is initialized, which takes some time

In [ ]:
print(f"Data Events: {data.nevents}\nExpected Events: {expectation_density.expected_counts():.2f}\nRelative Deviation {100 * (expectation_density.expected_counts()/data.nevents - 1):.3f} %")

Now you could save the cache.

In [ ]:
response.save_caches(data_path / "Crab_tutorial")

### Fitting

If the fit fails with "Current minimum stored after fit ... and current ... do not correspond!" simply rerun this cell (sometimes even that does not help).

In [ ]:
like.fit()

Now we can plot the result and compare it with the injected spectrum.

In [ ]:
results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["Crab"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["Crab"].spectrum.main.shape.evaluate_at, **parameters)

In [ ]:
energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)

In [ ]:
fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, energy**2 * flux_median, label = "Best fit")
ax.fill_between(energy, energy**2 * flux_lo, energy*energy*flux_hi, alpha = .5, label = "Best fit (errors)")
ax.plot(energy, energy**2 * flux_inj, color = 'black', ls = ":", label = "Injected")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$E^2 \frac{\mathrm{d}N}{\mathrm{d}E}$ [keV cm$^{-2}$ s$^{-1}$]")

ax.legend();